<a href="https://www.nvidia.com/dli"> <img src="images/DLI_Header.png" alt="Header" style="width: 400px;"/></a>

# 深度学习 —— 工业检测 #

## 01 - 自动光学检测和数据探索 ##
在这一部分，您将了解本课程背后的动机，并开始进行深度学习开发工作流的第一步，即数据探索。

**目录**
<br>
此 Notebook 包含以下几个部分：
1. [工业检测](#s1-1)
    * [制造业中的检测](#s1-1.1)
    * [PCBA 案例研究](#s1-1.2)
2. [机器学习工作流](#s1-2)
3. [数据集简介](#s1-3)
    * [数据收集](#s1-3.1)
    * [数据清洗](#s1-3.2)
    * [练习 #1 - 删除重复项](#s1-e1)
    * [数据探索](#s1-3.3)
    * [练习 #2 - 按元器件类型计数](#s1-e2)
    * [直观呈现数据集](#s1-3.4)
    * [练习 #3 - 绘制假阳性样本的图表](#s1-e3)
    * [练习 #4 - 绘制真阳性样本的图表](#s1-e4)
4. [确定项目范围](#s1-4)
    * [练习 #5 - 计算当前 AOI 误报率](#s1-e5)
5. [使用 DALI 进行数据预处理](#s1-5)
    * [DALI 流水线](#s1-5.1)
    * [数据增强](#s1-5.2)

<a name='s1-1'></a>
## 工业检测 ##
**工业检测**过程旨在防止不合格或不安全的产品到达客户手中。该领域在自动化和效率方面取得了重大进展。与过去完全由人工操作员进行的工作不同，现代检测涉及使用专用机械。在当今实际采用的不同类型的检测中，**光学检测**应用得最为广泛。为了高效进行光学检测，我们使用多种系统来捕获和比较图像，以大规模识别缺陷。我们使用的技术通常基于规则，其中涉及：
* 模板匹配 - 将捕获的图像与"黄金标准"进行比较。
* 模式匹配 - 将图像与合格样本和不合格样本进行匹配。
* 统计模式匹配 - 与模式匹配类似，但会使用一些样本，以容忍可接受的细微偏差

虽然这些技术显著改进了检测过程，但仍存在一些关系到可靠性的已知问题。随着**计算机视觉**技术的改进，基于深度学习的解决方案可以解决传统光学检测过程面临的挑战。虽然本课程主要侧重于制造业用例，但我们相信，开发过程经过调整后，可以应用于各种工业检测情况。

<a name='s1-1.1'></a>
### 制造业中的检测 ###
在制造领域，检测是在生产过程中进行的一项基本工作，有助于控制产品质量。我们通常会对每件产品进行检测，以确保离开生产线的每件产品都具有非常高的质量，并且没有制造缺陷。在产品制造过程中，由于种种原因，零部件和元器件难免会出现各种缺陷。这些缺陷不仅会影响产品的性能，甚至可能会危及安全。因此，未能执行质量保证检查可能会带来巨大的运营、财务和声誉风险。

<a name='s1-1.2'></a>
### PBCA 案例研究 ###
在开始开发之前，我们先讨论一下**印刷电路板组件 (PCBA)** 制造面临的问题。过去 50 年内，在半导体行业，[摩尔定律](https://www.darpa.mil/attachments/eri_design_proposers_day.pdf)一直在延续：在相同裸片面积内放置更多晶体管，在提高性能的同时降低成本。与此同时，PCBA 设计的复杂程度大大增加，尺寸则在不断缩小。这导致此类产品的成本增加，尤其是与生产验证相关的成本。
<p><img alt='Figure. The Curse of Moore’s Law - Source: DARPA,  Intelligent Design of Electronic Assets (IDEA)' src="images/darpa_moore.jpg" width=720></p>

在 PCBA、半导体晶圆、显示面板、电池等电子元器件的制造过程中，都会用到**自动光学检测 (AOI)** 机器。焊接过程完成后，便可在生产线中加入自动光学检测系统。及早发现问题可以降低成本，因为与在组装过程中及早发现缺陷相比，修复缺陷的成本通常会更高。此外，如果是与制造过程相关的系统性问题，我们需要及早发现它们。如果能够快速做出反应，便可以确保快速识别并纠正问题，从而避免制造出过多存在相同问题的产品。自动光学检测系统还需要灵活可靠，能够针对各种供应商和设计进行调整。

在分析电路板图像时，AOI 系统会查找各种具体缺陷，例如材料表面的瑕疵、焊接缺陷或元器件缺失/错位。AOI 机器检查产品时，会首先捕获**感兴趣区域 (ROI)** 的图像，然后"检查"是否存在任何缺陷。一个主要问题是，检查算法通常基于采用规则的传统计算机视觉方法，这通常会导致非常高的**假阳性**率。假阳性是指，自动化解决方案将实际上没有缺陷的产品识别为有缺陷的产品（即误报）。遗憾的是，在制造业中，假阳性率高 (> 10%) 是一个常见问题，因此要在制造过程中安排人工操作员进行验证。这非常具有挑战性，因为在验证过程中，需要由人工操作员进行繁琐的手动检测，为的是从假阳性的产品中分类出真正有缺陷的产品，而这会给生产的吞吐量带来不利影响。此外，手动检测很容易出错，因为人工操作员在每个电路板上所花的时间非常短，比如不到 30 秒，以便跟上生产速度。当人工操作员在长时间连续工作情况下感到疲劳时，问题会更加严重，可能会因此而漏报真正有缺陷的产品，这种情况称为**逃逸**。无论是 AOI 机器，还是人工检测员，只要漏掉了真正有缺陷的产品，就意味着发生了逃逸（即漏报）。
<p><img alt='Figure. Conventional PCBA Automated Optical Inspection Pipeline' src="images/PCBA_AOI.png" width=720></p>

缺陷检测过程会将产品分类为：
1. OK - 无缺陷（真阴性）
2. NG - 由人工操作员确认为无缺陷（假阳性）
3. NG - 由人工操作员确认为有缺陷（真阳性）
4. OK - 有缺陷/逃逸（假阴性）

<p><img alt='Figure. Automated Optical Inspection Flow' src='images/Manufacturing_AOI.png' width=720></p>

总而言之，当前流程面临以下几个挑战：
* 基于规则的传统 AOI 机器会产生非常高的假阳性率，而这会降低生产吞吐量并增加成本
* 缺乏灵活性，无法适应快速发展的零部件和产品设计
* 缺陷漏报造成的损失非常高

尽管在复杂性和/或吞吐量非常高的情况下，AOI比人工检测具有明显的优势，但在系统和软件开发方面，以及在工厂车间安排设备时，传统的图像处理系统和算法存在一些明显的缺点。进行自动光学检测之所以非常困难，是因为必须要检测大量的指标。在检测 PCBA 时，焊点的质量只是要检测的指标之一。此外，还必须要确认所有元器件是否都存在，以及每个元器件相对于阻焊层的位置和方向。其它指标包括元器件共面性、元器件是否被抬起，或者电路板表面是否存在不应存在的物体，例如焊料飞溅/焊球或其它污染物。针对所有情况和所有例外情况制定规则几乎是不可能的。我们可以使用基于深度学习的解决方案来解决这个问题。

<a name='s1-2'></a>
## 机器学习工作流 ##
我们将了解基于深度学习的检测过程具备的优势。除了能够准确检测缺陷之外，这种解决方案还可以降低验证成本，并提高生产吞吐量。具体来说，我们将演示端到端开发过程，以利用 NVIDIA 印刷电路板组件 (PCBA) 数据集改进缺陷检测过程，并使该过程实现自动化。同样的过程也适用于许多其它用例。
<p><img src='images/ml_workflow.png' width=720></p>

<a name='s1-3'></a>
## 数据集简介 ##
我们要使用的数据集来自 PCBA AOI 机器。关于前面讨论的验证过程，所有被视为有缺陷的图像（无论是假阳性还是真阳性）都将与元数据一起存储在 AOI 机器中。下图显示了所有相关数据，包括目录、html 和 xml 文件，以及图像。请注意，如果 AOI 机器未检测到缺陷，通常不会存储数据，因为存储此类数据不仅效率低下，而且会占用大量内存。我们将处理包含本课程所使用的数据的 zip 文件。
<p><img src="images/directory.png" width=720></p>

In [1]:
# DO NOT CHANGE THIS CELL
# import dependencies
import os

# set data path as environment variable
os.environ['DATA_PATH']='/dli/task/data'

# unzip
!unzip -qq $DATA_PATH/viz_BYD_new.zip -d data

# remove zip file
!rm $DATA_PATH/viz_BYD_new.zip

unzip:  cannot find or open /dli/task/data/viz_BYD_new.zip, /dli/task/data/viz_BYD_new.zip.zip or /dli/task/data/viz_BYD_new.zip.ZIP.
rm: /dli/task/data/viz_BYD_new.zip: No such file or directory


下面，我们来看看数据集的其中一个目录下的文件：

In [2]:
# DO NOT CHANGE THIS CELL
!ls -al $DATA_PATH/AOI_DL_data_0811/0422718064658

ls: /dli/task/data/AOI_DL_data_0811/0422718064658: No such file or directory


<a name='s1-3.1'></a>
### 数据收集 ###
在本课程中，我们将使用一些标准库来处理数据，其中包括 Pandas。Pandas [DataFrame](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.html) 是一种带标签的二维数据结构，其中可以容纳不同的数据类型。您可以将 DataFrame 想象成 Python 中用于存储、分析和操控数据的电子表格或类似于 SQL 的表格。请务必先熟悉 DataFrame，然后再继续学习下一部分。

In [3]:
# DO NOT CHANGE THIS CELL
# import dependencies
import pandas as pd
import re
import warnings
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import math
import numpy as np
import shutil
import time

warnings.filterwarnings("ignore")

现在，我们知道了数据是如何存储的，接下来我们要编写一些脚本来解析数据，为深度学习建模准备数据集。我们要定义一个称为 `parse_board_dir` 的函数，以便根据具有唯一性的序列号生成一个 **Pandas Series** 列表，其中包含来自指定目录的必要信息，例如图像路径、日期、图像形状等。`defect_image_path` 包含指向每个 PCBA 元器件的唯一位置的路径，`defect_image_name` 一般是为元器件类型指定的名称，并且会在不同电路板之间共用。下面，我们将收集数据目录中的所有 AOI 样本。这个过程最多可能需要几分钟才能完成。

元器件类型由元器件标识号的第一个字母表示。以下是 PCBA 中常见的元器件类型：
* **C** - 电容器
* **U** - 集成电路
* **Q** - 晶体管
* **R** - 电阻器
* **L** - 电感器
* **Y** - 振荡器
* **D** - 二极管
* **FL** - 滤波器
* **M** - 电机
* **J** - 插孔连接器
* **T** - 变压器

In [4]:
# DO NOT CHANGE THIS CELL
# define function to parse a board directory
def parse_board_dir(dir_path, date): 
    row_list=[]
    true_defect='notdefect'
    board=dir_path.split('/')[-1]
    for file in os.listdir(dir_path): 
        if re.match("^D\d{1}_", file): 
            row={'true_defect': true_defect, 
                 'defect_img_path': os.path.join(dir_path, file), 
                 'defect_image_name': file, 
                 'img_shape': mpimg.imread(os.path.join(dir_path, file)).shape, 
                 'board': board, 
                 'comp_id': file.split('_')[-1].split('.')[0], 
                 'comp_type': re.findall('_(\D+)', file)[0],
                 'date': date}
            row_list.append(row)
    return pd.DataFrame(row_list)

In [5]:
# DO NOT CHANGE THIS CELL
# time the process
start=time.time()

# create empty list
date_list=[]

# iterate through each date directory
for date_dir in os.listdir(os.environ['DATA_PATH']): 
    date_dir_path=os.path.join(os.environ['DATA_PATH'], date_dir)
    if os.path.isdir(date_dir_path): 
        date=date_dir.split('_')[-1]
        for board_dir in os.listdir(date_dir_path): 
            board_dir_path=os.path.join(os.environ['DATA_PATH'], date_dir, board_dir)
            if os.path.isdir(board_dir_path): 
                # add data to list
                date_list.append(parse_board_dir(board_dir_path, date))

# create dataframe
df=pd.concat(date_list, ignore_index=True)

print('It took {} seconds to gather {} images.'.format(round(time.time()-start, 2), len(df)))

FileNotFoundError: [Errno 2] No such file or directory: '/dli/task/data'

下面显示的 DataFrame 中包含所有图像的清单。

In [ ]:
# DO NOT CHANGE THIS CELL
# preview dataframe
df.head()

<p><img src='images/important.png' width=720></p>

此图像列表包含被 AOI 机器视为有缺陷的图像，但其中既包括真阳性图像，也包括假阳性图像。对于来自生产线的产品，仅当 AOI 机器将其图像分类为有缺陷时，才会存储相应图像。如果存储生产过程中被检测的每个零部件的图像，将需要大量的存储空间，而这是不可行的。对于被 AOI 机器认为没有缺陷的元器件，我们没有它们的图像。不过，若要构建用于检测缺陷的机器学习分类器，我们既需要有缺陷的图像，也需要没有缺陷的图像。在我们拥有的图像中，人工操作员会手动识别真阳性缺陷，并在单独的文件中跟踪它们。下图显示了一个示例文件，其中列出了真阳性元器件及其各自的图像。对于每个星期，此文件都将存储为 `AOI Defect list.xlsx`。我们将解析此跟踪文件，并修改这些图像的 `true_defect` 指示标记。在此示例中，我们将使用被人工检测员判定为假阳性的图像作为**无缺陷图像** (`notdefect`)，并使用被人工检测员判定为真阳性的图像作为**有缺陷图像** (`defect`)，以便进行机器学习。
<p><img src='images/true_positives.jpg' width=720></p>

In [ ]:
# DO NOT CHANGE THIS CELL
# time the process
start=time.time()

# iterate through each date directory
for date_dir in os.listdir(os.environ['DATA_PATH']): 
    date_dir_path=os.path.join(os.environ['DATA_PATH'], date_dir)
    date=date_dir.split('_')[-1]
    
    # look for defect list spreadsheet
    if os.path.exists(os.path.join(date_dir_path, 'AOI Defect list.xlsx')): 
        defect_df=pd.read_excel(os.path.join(date_dir_path, 'AOI Defect list.xlsx'))
        defect_df['date']=date
        
        for idx, row in defect_df.iterrows(): 
            defect_list=row['Defect Location'].split(',')
            for defect in defect_list: 
                # search for matching record
                found_df=df.loc[(df['date']==date) & (df['board']==('0'+str(row['SN']))) & (df['defect_image_name']==(defect+'.jpg'))]
                df.loc[(df['date']==date) & (df['board']==('0'+str(row['SN']))) & (df['defect_image_name']==(defect+'.jpg')), 'true_defect']='defect'
                if len(found_df)==0:
                    print(f"{date} | {row['SN']} | {row['Defect Location']} not found")
        
print('It took {} seconds to process the metadata for {} defective images.'.format(round(time.time()-start, 2), len(df[df['true_defect']=='defect'])))

<a name='s1-3.2'></a>
### 数据清洗 ###
我们可以执行一些基本的数据质量检查，以便确定：
1. 是否有重复项？ 
2. 对于每个电路板，是否有任何元器件具有混合的缺陷状态？ 

如果存在重复项，则表示相应数据应予以删除。下面，我们将根据 `board` 和 `comp_id` 来统计重复条目的数量。有些元器件在每个电路板上需要有多个。

In [ ]:
# DO NOT CHANGE THIS CELL
# group rows by board serial number and component id
df.groupby(['board', 'comp_id'])['defect_image_name'] \
  .apply(list) \
  .sort_values(key=lambda x: x.str.len(), ascending=False)

<a name='s1-e1'></a>
### 练习 #1 - 删除重复项 ###

**说明**：<br>
* 执行以下单元，以便根据 `board` 和 `defect_image_name` 生成包含重复条目的 DataFrame，并预览该 DataFrame。对于每个电路板，图像名称都应该是独一无二的。
* 仅修改 `<FIXME>`，并执行以下单元，以便统计重复的条目的数量。
* 回答以下单元中的问题。
* 执行以下单元，以便删除重复的行，这些行代表了在不同日期被复制的相同的条目。

In [ ]:
# DO NOT CHANGE THIS CELL
# check for duplicates
duplicated_df=df[df.duplicated(['board', 'defect_image_name'], keep=False)]
duplicated_df.sort_values(['board', 'comp_id']) \
             .head()

In [ ]:
len(<<<<FIXME>>>>)

In [ ]:
# DO NOT CHANGE THIS CELL
# delete duplicates
df=df.drop_duplicates(['board', 'defect_image_name'])
print('There are {} rows left'.format(len(df)))

单击 ... 即可显示**解决方案**。

我们要将 DataFrame 的副本保存为 `pcba_df.csv`，以备后用。

In [ ]:
# DO NOT CHANGE THIS CELL
# save a copy of dataframe
df.to_csv('pcba_df.csv', index=False)

<a name='s1-3.3'></a>
### 数据探索 ###
我们将进行一些探索性数据分析，以便确定：
1. 在被 AOI 机器检测为有缺陷的元器件中，有多少确实有缺陷（真阳性），有多少实际上并没有缺陷（假阳性）？ 是否存在数据不平衡问题？ 
2. 随着时间的推移，缺陷总数有何变化？ 
3. 哪种元器件的缺陷最多？ 哪种元器件的假阳性缺陷最多？ 

我们先来看看真阳性缺陷与假阳性缺陷。

In [ ]:
# DO NOT CHANGE THIS CELL
# plot bar graph of percent of true positive and false positive
df['true_defect'].value_counts(normalize=True) \
                 .plot(kind='bar', 
                       figsize=(5, 5), 
                       title='False Positive vs. True Positive', 
                       color=['g', 'r'])

通过将假阳性缺陷和真阳性缺陷一起绘制在图表中，我们可以清楚地看到，真正的缺陷在数据中所占的比例要小得多。

<img src='images/tip.png' width=720>

在实际生产中，**dppm（每百万零部件中的缺陷零部件）**率通常低于 100 dppm，对于安全性至关重要的应用领域，通常低于 10 dppm。虽然我们无法在不知道生产了多少产品的情况下确定 dppm，但我们的目标应低于 1 dppm，尤其是对于安全性至关重要的应用领域，例如汽车行业。借助人工智能的强大功能，我们可以准确高效地识别有缺陷的产品，然后将其送修。

下面，我们来看看缺陷数随时间变化的情况。

In [ ]:
# DO NOT CHANGE THIS CELL
# plot total defect count by time
fig, ax=plt.subplots(1, 2, figsize=(10, 5))
df.groupby('date') \
  .size() \
  .plot(ax=ax[0])

# plot true positive and false positive count by time
date_df=df.pivot_table(index='date', columns='true_defect', aggfunc='size', fill_value=0)
date_df.plot(ax=ax[1], color=['r', 'g'])

随着时间的推移，潜在缺陷的总数在不断变化，而真正缺陷的数量则一直保持在**较低水平**。这凸显了基于规则的 AOI 算法面临的一些挑战，因为这些算法高度敏感，并且需要不断调整。数据表明，AOI 机器在 9 月 30 日检测出的潜在缺陷的数量异常高，在 10 月 27 日检测出的潜在缺陷的数量则异常低。虽然这些问题很快得到解决，但这意味着人工操作员要承受不可预测的工作量的影响。

我们还可以直观呈现，有百分之多少的潜在缺陷被人工操作员确认为真正的缺陷。

In [ ]:
# DO NOT CHANGE THIS CELL
# plot true positive and false positive percentage over time
date_df.div(date_df.sum(axis=1), axis=0).plot(kind='bar', figsize=(5, 5), stacked=True, color=['r', 'g'])
plt.legend(loc='upper right')

<a name='s1-e2'></a>
### 练习 #2 - 按元器件类型计数 ###
我们来看看哪种元器件的缺陷最多。

**说明**：<br>
* 仅修改 `<FIXME>`，并执行以下单元，以便确定哪 5 种元器件被 AOI 机器检测出的缺陷最多。
* 执行以下单元，以便按元器件类型绘制缺陷数量图表。

In [ ]:
# group by first letter of component id and plot the number of records
top_five_component_types=df.groupby(<<<<FIXME>>>>) \
                           .size() \
                           .sort_values(ascending=False) \
                           .head(5) \
                           .index

print('The top 5 component types are: {}'.format(list(top_five_component_types)))

In [ ]:
# DO NOT CHANGE THIS CELL
# filter dataframe by top 5 component types
top_five_component_types_df=df[df['comp_type'].isin(top_five_component_types)]

# plot defect counts by type
top_five_component_types_df.groupby('comp_type') \
                           .size() \
                           .plot(kind='bar', 
                                 figsize=(5, 5), 
                                 title='Number of Components Tested Positive by Type')

单击 ... 即可显示**解决方案**。

如果按元器件类型显示不同时间的缺陷总数，我们可以观察到一个有趣的动态情况。

In [ ]:
# DO NOT CHANGE THIS CELL
# pivot dataframe by date and component type to see the trend
top_five_component_types_df=df[df['comp_type'].isin(top_five_component_types)]
top_five_component_types_df.pivot_table(index='date', columns=df['comp_type'], aggfunc='size', fill_value=0) \
                           .plot(figsize=(5, 5), 
                                 title='Weekly True Positive Defect Count by Component Type')

<p><img src='images/important.png' width=720></p>

根据以上内容，我们可以看到，潜在缺陷数量随时间变化的情况可能会因元器件类型而异。AOI 机器检测出的潜在缺陷元器件越来越多的原因可能有很多，包括供应商、制造情况或 AOI 机器本身。为了适应环境变化，需要根据业务需求对 AOI 机器的参数不断进行调整。例如，我们在 9 月 30 日观察到电容器和晶体管的检测数量下降，这可能是导致集成电路和电阻器的检测数量上升的原因。对于任何生产线来说，这都是一个现实的情况。

我们还可以查看类似信息，但这些信息会按真阳性与假阳性区分开来。

In [ ]:
# DO NOT CHANGE THIS CELL
# group by component type and plot true positive vs. false positive
top_five_component_types_df.pivot_table(index=df['comp_type'], columns='true_defect', aggfunc='size') \
                           .plot(kind='bar', 
                                 stacked=True, 
                                 figsize=(5, 5), 
                                 color=['r', 'g'])

In [ ]:
# DO NOT CHANGE THIS CELL
# pivot dataframe by date and component type to see the trend
top_five_component_types_df[top_five_component_types_df['true_defect']=='defect'].pivot_table(index='date', columns=df['comp_type'], aggfunc='size', fill_value=0) \
                                                                                 .plot(figsize=(5, 5), 
                                                                                       title='Weekly True Positive Defect Count by Component Type')

电容器报告了大量缺陷。

<a name='s1-3.4'></a>
### 直观呈现数据集 ###
我们来预览一些图像，看看能不能找出一些有助于识别电容器是否存在缺陷的明确模式。下面的 `initial_spot` 函数用于绘制给定 DataFrame 中的样本图像。

In [ ]:
# DO NOT CHANGE THIS CELL
# define function plot sample df
def view_samples(pcba_df):
    '''
    plot unique images to check how true positives or false positive images look like in the provided dataframe 
    (to get a feeling on whether it make sense to develop a DL algorithm to classify them)
    
    param: 
        pcba_df: the original dataframe
    return: 
        none
    '''
    plt.figure(figsize=(30, 30))
    num_cols = 10
    # get the number of unique capacitors based on the component name (ID)
    num_unique_cap = len(pcba_df['comp_id'].unique())
    num_rows = math.ceil(num_unique_cap / num_cols)
    
    # use mpimg.imread to read the extracted image path
    # and plt.imshow to display image inside the subplot.
    for index, c_name in enumerate(pcba_df['comp_id'].unique()):
        image_path = pcba_df[pcba_df['comp_id'] == c_name]['defect_img_path'].values[0]
        plt.subplot(num_cols, num_rows, index+1)
        # assign the image name to the subplot
        plt.title(c_name)
        plt.imshow(mpimg.imread(image_path))
        plt.axis('off')

<a name='s1-e3'></a>
###  练习 #3 - 显示假阳性样本的图像 ###
我们来预览一些假阳性样本。

**说明**：<br>
* 仅修改 `<FIXME>`，并执行以下单元，以便显示一些假阳性样本的图像。
* 请注意以下单元中的假阳性样本是否存在任何常见的视觉主题。

In [ ]:
view_samples(<<<<FIXME>>>>)

单击 ... 即可显示**解决方案**。

<a name='s1-e4'></a>
### 练习 #4 - 显示真阳性样本的图像 ###
我们来预览一些真阳性样本。

**说明**：<br>
* 仅修改 `<FIXME>`，并执行以下单元，以便显示一些真阳性样本的图像。
* 请注意以下单元中的真阳性样本是否存在任何常见的视觉主题。

In [ ]:
view_samples(<<<<FIXME>>>>)

单击 ... 即可显示**解决方案**。

<a name='s1-4'></a>
## 确定项目范围 ##
在开始模型开发过程之前，必须先确定项目范围，以便在短时间内（例如 3 到 6 个月内）就能测试方案的可行性。理想情况下，我们希望构建一个能够对 PCBA 案例中所有元器件的缺陷进行分类的模型。不过，制定一个项目来处理所有元器件和所有缺陷类型可能是一个过高的目标，需要较长时间才能获得有希望的结果，这可能不符合项目的时间表和限制条件，并且可能需要大量的管理工作。此外，AI 项目通常涉及多个学科，需要包括 AOI 技术人员/工程师、数据科学家和 AI 工程师在内的现场操作团队通力协作，以获得有意义的结果。同时还需要熟悉缺陷模式的人工操作员来标记和记录真正的缺陷，而这需要大量的时间和耐心。

NVIDIA 的制造团队向开发团队提出要求，希望他们专注于想办法利用深度学习来降低电容器的误报率。在项目规划期间，我们务必要注重倾听内部客户的心声，以便获得管理层对数据收集活动和资源分配的支持。在本示例中，我们将重点关注**电容器缺陷检测**自动化，它导致的假阳性率远高于印刷电路板上的其它元器件。下面是一个放大后的PCBA上的电容器照片。
<p><img alt='Figure. Capacitors in PCBA' src="images/Capacitor_PCB.jpg" width=720></p>

接下来，我们看看仅包含**电容器**的新 DataFrame。

<p><img src='images/tip.png' width=720></p>
虽然我们的项目范围仅限于电容器，但在完成本课程后，您可以考虑查看其它元器件类型。集成电路 (U) 非常适合用来练习以磨练您的技能，而其它元器件类型可能缺少构建可靠的深度学习模型所需的数据量。

In [ ]:
# DO NOT CHANGE THIS CELL
# filter the dataframe to only include Capacitors
capacitor_df=df[df['comp_type']=='C']

# preview capacitor dataframe
capacitor_df.head()

<a name='s1-e5'></a>
### 练习 #5 - 计算当前 AOI 误报率 ###
我们来计算一下，对于电容器，在当前的 AOI 机器报告的潜在缺陷中，有百分之多少是假阳性。

**说明**：<br>
* 仅修改 `<FIXME>`，并执行以下单元，以便生成标注了错误率的条形图。
* 请回答以下单元中关于[错误发现率](https://en.wikipedia.org/wiki/False_discovery_rate)的问题。

In [ ]:
# display the normalized counts of false positive and true positive for capacitors
display(capacitor_df[<<<<FIXME>>>>].value_counts(normalize=True))

# plot the distribution of false positive and true positive for capacitors
capacitor_df[<<<<FIXME>>>>].value_counts(normalize=True) \
                           .plot(kind='bar', 
                                 figsize=(5, 5), 
                                 title='False Positive vs. True Positive', 
                                 color=['g', 'r'])

单击 ... 即可显示**解决方案**。

<p><img src='images/important.png' width=720></p>

潜在缺陷中真阳性缺陷所占的百分比并不能反映缺陷率。为了计算缺陷率，我们还需要知道 PCBA 生产总量。

我们要将 DataFrame 的副本保存为 `capacitors_df.csv`，以备后用。

In [ ]:
# DO NOT CHANGE THIS CELL
# save a copy of dataframe
capacitor_df.to_csv('capacitor_df.csv', index=False)

<p><img src='images/tip.png' width=720></p>

与这个 DLI 示例不同，实际来自生产线的各种传感器输出的 Excel 数据文件很容易超过 100 GB。加载这种数据文件可能会占用大量 CPU 时间。NVIDIA 的 RAPIDS 是一个开源软件库，适用于端到端数据科学和机器学习分析流水线。请不妨花点时间访问 [RAPIDS](https://rapids.ai/)。

<a name='s1-5'></a>
## 使用 DALI 进行数据预处理 ##
深度学习模型需要大量数据才能生成准确的预测，而随着模型规模和复杂性不断增加，这一需求变得越来越强烈。无论什么模型，都需要进行一定程度的数据预处理，以便进行训练和推理。在计算机视觉应用中，预处理通常包括解码、调整大小，以及归一化为神经网络接受的标准化格式。以前，深度学习工作负载的数据预处理一直没有引起太多关注，而训练复杂模型所需的巨大计算资源则凸显了数据预处理的重要性。这些预处理例程通常称为流水线，目前使用 OpenCV、Pillow 等库在 CPU 上执行。当今的 深度学习应用包括复杂的多阶段数据处理流水线，而这些流水线则包含许多按顺序执行的操作。依靠 CPU 来处理这些流水线已成为制约性能和可扩展性的瓶颈。
<p><img src='images/dali.png' width=720></p>

**NVIDIA 数据加载库** (DALI) 是一个用于加载和预处理数据的库，可加速深度学习应用。该库提供了一系列高度优化的构建块，以便加载和处理图像、视频和音频数据。DALI 通过将数据预处理卸载到 GPU 来解决 CPU 瓶颈问题。此外，它还提供了一些强大的功能：
* DALI 为各种深度学习应用提供数据处理原语。支持的输入格式包括各种最常用的图像文件格式。
* DALI 依赖于自己的执行引擎，该引擎旨在更大限度地提高输入流水线的吞吐量。
* 它可以用作可移植的插件，以取代热门深度学习框架中的内置数据加载器和数据迭代器。
* 采用对用户透明的方式处理预取、并行执行和批处理等功能。
* 不同的深度学习框架有多种数据预处理实现方式，这带来了一些挑战，例如训练和推理工作流的可移植性，以及代码的可维护性。使用 DALI 实现的数据处理流水线是可移植的，因为可以轻松地将它们再迁至 TensorFlow、PyTorch、MXNet 和 PaddlePaddle。
* 通常，用于推理的预处理例程与用于训练的例程类似，因此，使用相同的工具实现这两者可减少所需的样板和代码重复。

<a name='s1-5.1'></a>
###  DALI 流水线 ###
使用 DALI 处理数据的核心是数据处理 `pipeline` 的概念。它由有向图中多个相连的操作组成，包含在 `nvidia.dali.Pipeline` 类的对象内。这个类提供了定义、构建和运行数据处理流水线所必需的函数。流水线中的每个操作通常都会接收一个或多个输入，应用某种数据处理操作，然后生成一个或多个输出。有些特殊类型的操作不接收任何输入，也不生成输出。这些特殊操作的行为方式类似于数据源，读取器、随机数生成器和外部来源都属于这个类别。

DALI 为多种处理的操作提供 CPU 和 GPU 实现。可以使用哪种实现取决于操作的性质。请务必查看文档，了解[最新的受支持的操作列表](https://docs.nvidia.com/deeplearning/dali/user-guide/docs/#operations)，该列表会随着每个版本的发布而扩展。若要定义 DALI 流水线，最简易的方式是使用 `pipeline_def` Python [装饰器](https://peps.python.org/pep-0318/)。为了创建流水线，我们需要定义一个函数，以便实例化和连接所需的操作，并返回相关输出。然后，只需使用 `pipeline_def` 装饰它即可。我们先来定义一个非常简单的流水线，它将包含两个操作。第一个操作是文件读取器 (`fn.readers.file`)，用于发现和加载目录中包含的文件。读取器用于输出文件内容（本例中为 JPG 图像）和标签。文件读取器可用于根据目录结构来推理“文件-标签”对。第二个操作是[图像解码器](https://docs.nvidia.com/deeplearning/dali/user-guide/docs/supported_ops.html#nvidia.dali.fn.decoders.image) (`fn.decoders.image`)。最后，我们会返回“图像-标签”对。在 `simple_pipeline` 函数中，我们定义了要执行的操作以及它们之间的计算流。如需详细了解 `pipeline_def`，请查看[文档](https://docs.nvidia.com/deeplearning/dali/user-guide/docs/pipeline.html?#nvidia.dali.pipeline_def)。

In [ ]:
# DO NOT CHANGE THIS CELL
# import dependencies
from matplotlib import gridspec
from nvidia.dali.pipeline import Pipeline
from nvidia.dali import pipeline_def
import nvidia.dali.fn as fn
import nvidia.dali.types as types
from PIL import Image
import warnings

warnings.filterwarnings("ignore")

In [ ]:
# DO NOT CHANGE THIS CELL
batch_size=8
defect_label_map={'notdefect': 1, 'defect': 0}
defect_inverse_map={v: k for k, v in defect_label_map.items()}
sample=capacitor_df.groupby('true_defect', group_keys=False).apply(lambda x: x.sample(n=batch_size))

@pipeline_def
def simple_pipeline():
    # use fn.readers.file to read encoded images and labels from the hard drive
    jpgs, labels=fn.readers.file(files=sample['defect_img_path'].to_list(), labels=sample['true_defect'].map(defect_label_map).to_list())
    # use the fn.decoders.image operation to decode images from JPG to RGB
    images=fn.decoders.image(jpgs, device='cpu')
    # specify which of the intermediate variables should be returned as the outputs of the pipeline
    return images, labels

为了使用通过 `simple_pipeline` 定义的流水线，我们需要创建并构建它。这是通过调用 `simple_pipeline()` 实现的，此类调用会创建流水线的实例。然后，我们针对这个新创建的实例调用 `build()`：

In [ ]:
# DO NOT CHANGE THIS CELL
# create and build pipeline
pipe=simple_pipeline(batch_size=batch_size*2, num_threads=4, device_id=0)
pipe.build()

<p><img src='images/important.png' width=720></p>

请注意，使用 `pipeline_def` 装饰函数会向函数添加新的命名参数。这些参数可用于控制流水线的各个方面，例如批量大小、用于在 CPU 上执行计算的线程数，以及要使用的 GPU 设备（尽管使用 simple_pipeline 创建的流水线尚未使用 GPU 进行计算）。如需详细了解 `Pipeline` 参数，您可以查看[流水线文档](https://docs.nvidia.com/deeplearning/dali/user-guide/docs/pipeline.html)。

构建完毕后，流水线实例会通过调用流水线的 `run()` 方法来获取批量结果，从而以[异步](https://en.wikipedia.org/wiki/Asynchrony_(computer_programming)方式运行。我们则按照我们希望的那样，把结果拆解成 `images` 和 `labels`这两种元素，它们各自都有一个张量列表。

In [ ]:
# DO NOT CHANGE THIS CELL
# run the pipeline
simple_pipe_output=pipe.run()

images, labels=simple_pipe_output
print("Images is_dense_tensor: " + str(images.is_dense_tensor()))
print("Labels is_dense_tensor: " + str(labels.is_dense_tensor()))

为了查看图像，我们需要使用对应的 `at` 方法循环访问 `TensorList` 中包含的所有张量。

In [ ]:
# DO NOT CHANGE THIS CELL
# define a function display images
def show_images(image_batch, label_batch):
    columns=4
    rows=math.ceil(len(image_batch)/columns)
    # create plot
    fig=plt.figure(figsize=(10, (10 // columns) * rows))
    gs=gridspec.GridSpec(rows, columns)
    for idx in range(rows*columns):
        plt.subplot(gs[idx])
        plt.axis("off")
        plt.imshow(image_batch.at(idx))
        plt.title(defect_inverse_map[label_batch.at(idx)[0]])
    plt.tight_layout()

show_images(images, labels)

<a name='s1-5.2'></a>
### 数据增强 ###
深度学习模型需要使用大量的数据进行训练，以获得准确的结果。DALI 不仅能够读取磁盘中的图像，将其批处理为张量，还能够对这些图像进行各种增强，以改进深度学习训练结果。[数据增强](https://en.wikipedia.org/wiki/Data_augmentation)是指对数据引入随机干扰（例如几何变形、转换颜色、添加噪点等），从而人为地增加数据集的大小。这些干扰有助于生成在预测方面更可靠的模型，避免过拟合，并实现更高的准确性。我们将使用 DALI 来演示为了进行模型训练而引入的数据增强，例如裁剪、调整大小和翻转。
<p><img src='images/augmentation.png' width=720></p>

In [ ]:
# DO NOT CHANGE THIS CELL
@pipeline_def
def augmentation_pipeline():
    # use fn.readers.file to read encoded images and labels from the hard drive
    jpgs, labels=fn.readers.file(files=sample['defect_img_path'].to_list(), labels=sample['true_defect'].map(defect_label_map).to_list())
    # use the fn.decoders.image operation to decode images from JPG to RGB
    images=fn.decoders.image(jpgs, device='cpu')
    # use the fn.rotate operation to rotate image
    rotated_images = fn.rotate(images.gpu(), angle=45, fill_value=0)
    return rotated_images, labels

In [ ]:
# DO NOT CHANGE THIS CELL
augmentation_pipe=augmentation_pipeline(batch_size=batch_size*2, num_threads=4, device_id=0)
augmentation_pipe.build()

<p><img src='images/important.png' width=720></p>

DALI 不支持在流水线中将数据从 GPU 迁移到 CPU。因此，GPU 操作后无法紧跟 CPU 操作。`augmentation_pipe_output` 中包含 2 个 TensorList，其中一个是在 GPU 上执行的 `rotate` 操作的结果。由于我们不能直接从 CPU 访问 `TensorListGPU` 的内容，因此我们需要使用 `as_cpu` 方法将其复制到 CPU，以便显示结果。

In [ ]:
# DO NOT CHANGE THIS CELL
augmented_images, labels=augmentation_pipe.run()
show_images(augmented_images.as_cpu(), labels)

**非常棒！** 完成本课后，请先完成评估测验，然后再继续下一个实验。

<a href="https://www.nvidia.com/dli"> <img src="images/DLI_Header.png" alt="Header" style="width: 400px;"/></a>